# Проверки `c_nazn` в `ods.scd1_z_main_docum`

Цели:
1. Посмотреть все уникальные варианты `c_nazn` за первую неделю июня.
2. Выделить варианты `c_nazn`, которые могут относиться к эквайрингу.

Период задается параметрами ниже (по умолчанию: с `2026-06-01` по `2026-06-07` включительно).

In [ ]:
import os
import pandas as pd
from rail_connectors.connection import connect


print('Imports loaded')

In [ ]:
# Параметры периода: первая неделя июня
week_start = '2026-06-01'
week_end_exclusive = '2026-06-08'  # c 01 по 07 июня включительно

# Параметры исполнения
mem_limit = '8g'
preview_limit = 500
load_all_unique = False
load_all_ekv = False

table_name = 'ods.scd1_z_main_docum'

print(f'Period: [{week_start}, {week_end_exclusive})')
print(f'table={table_name}')

In [ ]:
try:
    from connection_secrets import LAKE_USER, LAKE_PASSWORD
except Exception as exc:
    raise RuntimeError('Cannot import LAKE_USER/LAKE_PASSWORD from connection_secrets.py') from exc

keytab_path = '/home/jovyan/test_requests/tech.keytab'
kerberos_cfg = {
    'use_credentials': True,
    'update_keytab': False,
}

if os.path.exists(keytab_path):
    kerberos_cfg['keytab_path'] = keytab_path
    kerberos_cfg['update_keytab'] = True
else:
    print(f'WARNING: keytab not found: {keytab_path}')
    print('Using current ticket cache (kinit must be valid).')

imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos=kerberos_cfg,
    user_params={
        'user_name': LAKE_USER,
        'password': LAKE_PASSWORD,
    }
)
imp._init_connection()
print('Impala connection initialized')

## 1) Все уникальные варианты `c_nazn` за первую неделю июня

In [ ]:
sql_unique_stats = f"""
select
    count(*) as total_rows,
    count(distinct coalesce(c_nazn, '')) as unique_c_nazn_count
from {table_name}
where cast(c_date_prov as date) >= date '{week_start}'
  and cast(c_date_prov as date) < date '{week_end_exclusive}'
"""

with imp:
    imp.execute(f'set MEM_LIMIT={mem_limit}')
    unique_stats_df = imp.fetch(sql_unique_stats)

unique_stats_df

In [ ]:
sql_unique_variants_base = f"""
select
    coalesce(c_nazn, '') as c_nazn,
    count(*) as cnt
from {table_name}
where cast(c_date_prov as date) >= date '{week_start}'
  and cast(c_date_prov as date) < date '{week_end_exclusive}'
group by coalesce(c_nazn, '')
order by cnt desc
"""

sql_unique_variants = (
    sql_unique_variants_base
    if load_all_unique
    else sql_unique_variants_base + f'\nlimit {preview_limit}'
)

with imp:
    imp.execute(f'set MEM_LIMIT={mem_limit}')
    unique_variants_df = imp.fetch(sql_unique_variants)

print(f'Rows loaded: {len(unique_variants_df):,}')
unique_variants_df.head(50)

## 2) Варианты `c_nazn`, которые могут относиться к эквайрингу

In [ ]:
# Ловим слова от корня "эквайр" (эквайринг, эквайринга, эквайринговый и т.д.)
ekv_pattern = r'(^|[^а-яa-z0-9])(эквайр[а-я]*)([^а-яa-z0-9]|$)'

sql_ekv_stats = f"""
with scoped as (
    select regexp_replace(lower(coalesce(c_nazn, '')), 'ё', 'е') as nazn_norm
    from {table_name}
    where cast(c_date_prov as date) >= date '{week_start}'
      and cast(c_date_prov as date) < date '{week_end_exclusive}'
)
select
    count(*) as scoped_rows,
    sum(case when nazn_norm rlike '{ekv_pattern}' then 1 else 0 end) as matched_rows
from scoped
"""

with imp:
    imp.execute(f'set MEM_LIMIT={mem_limit}')
    ekv_stats_df = imp.fetch(sql_ekv_stats)

ekv_stats_df

In [ ]:
sql_ekv_variants_base = f"""
select
    coalesce(c_nazn, '') as c_nazn,
    count(*) as cnt
from {table_name}
where cast(c_date_prov as date) >= date '{week_start}'
  and cast(c_date_prov as date) < date '{week_end_exclusive}'
  and regexp_replace(lower(coalesce(c_nazn, '')), 'ё', 'е') rlike '{ekv_pattern}'
group by coalesce(c_nazn, '')
order by cnt desc
"""

sql_ekv_variants = (
    sql_ekv_variants_base
    if load_all_ekv
    else sql_ekv_variants_base + f'\nlimit {preview_limit}'
)

with imp:
    imp.execute(f'set MEM_LIMIT={mem_limit}')
    ekv_variants_df = imp.fetch(sql_ekv_variants)

print(f'Rows loaded: {len(ekv_variants_df):,}')
ekv_variants_df.head(100)

In [ ]:
# Необязательно: сохранить результаты в CSV
save_to_csv = False
unique_out_path = './c_nazn_unique_first_week_june.csv'
ekv_out_path = './c_nazn_ekv_first_week_june.csv'

if save_to_csv:
    unique_variants_df.to_csv(unique_out_path, index=False)
    ekv_variants_df.to_csv(ekv_out_path, index=False)
    print(f'Saved: {unique_out_path}')
    print(f'Saved: {ekv_out_path}')
else:
    print('save_to_csv=False, nothing was written.')